#   `lesson04`:  Image Processing II

In `lesson02`, you saw how Python can handle image data effectively to identify and modify features of MRI scan data.  This time around, you will apply data mining techniques to identify features of satellite images and astronomical images.

**Objectives:**
-   Convert an image between color spaces.
-   **Apply data mining techniques such as clustering to identify image features.**
-   Apply machine learning techniques to identify image features.


This lesson is divided into two parts:

1.  **(Class)**  We will cover data mining for image analysis.
2.  **(Individual)**  You will apply machine learning to identify image content.

In [ ]:
import numpy as np
from numpy import random as npr
import matplotlib.pyplot as plt
import matplotlib.cm as cm
%matplotlib inline

##  Image Analysis for Satellite Data

### Clustering Analysis

Clustering analysis is a type of unsupervised learning.  Two common methods will be examined:  $k$-means and hierarchical clustering.  In this section, we will be introducing the $k$-means method as well as criteria to evaluate the performance of clustering.

We will generate a clean random data set to demonstrate the technique initially.

In [ ]:
k = 3       # number of clusters
n = 50      # number of points per cluster

sigma = 0.4 # stdev of cluster
centers = [ ( 0,1 ),( 1,0 ),( -0.5,-0.5 ) ]

samples = np.zeros( ( k*n,2 ) )
for index,center in enumerate( centers ):
    samples[ index*n:( index+1 )*n,: ] = npr.randn( n,2 ) * sigma + center

fig,ax = plt.subplots()
for index,mrkr in enumerate( [ 'bo','ro','go' ] ):
    data = samples[ index*n:( index+1 )*n,: ]
    ax.plot( data[ :,0 ],data[ :,1 ],mrkr )
plt.title( 'Original Data' )
plt.show()

fig,ax = plt.subplots()
plt.plot( samples[ :,0 ],samples[ :,1 ],'ko' )
plt.title( 'Anonymized Data' )
plt.show()

#### k-means
[$k$-means clustering](https://en.wikipedia.org/wiki/K-means_clustering) takes a set of observations or samples and attempts to partition them into $k$ groups or clusters.  Obviously different values of $k$ will yield different clusters.

The algorithm:
$k$ centroids are randomly chosen in the domain.  (Specifically, the algorithm is reinitialized several times with random centroids and the best fit of several is returned.)  Clusters are created by associating every observation with the nearest centroid.  The new centroid of this updated cluster is then used, and the process proceeds iteratively to convergence.  Conveniently, cluster centroids can be suggested as well to "seed" the algorithm with suggestions from a human observer.

Although this algorithm isn't too bad to implement, [scikit-learn](https://scikit-learn.org/stable/), a python library for machine learning, conveniently implements this ([sklearn.cluster.KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)) and many other machine learning methods.

**$k = 2$**:

In [ ]:
from sklearn.cluster import KMeans

k = 2
est = KMeans( n_clusters=k )
est.fit( samples )
labels = est.labels_

fig,ax = plt.subplots()
data0 = samples[ labels == 0 ]
ax.plot( data0[ :,0 ],data0[ :,1 ],'bo' )
data1 = samples[ labels == 1 ]
ax.plot( data1[ :,0 ],data1[ :,1 ],'ro' )
plt.title( 'Clustered data, k = %i'%k )
plt.show()

fig,ax = plt.subplots()
ax.plot( data0[ :,0 ],data0[ :,1 ],'bo' )
ax.plot( data1[ :,0 ],data1[ :,1 ],'ro' )
for index,mrkr in enumerate( [ 'bx','rx','gx' ] ):
    data = samples[ index*n:( index+1 )*n,: ]
    ax.plot( data[ :,0 ],data[ :,1 ],mrkr )
plt.title( 'Clustered data, k = %i'%k )
plt.show()


**$k = 3$**:

In [ ]:
k = 3
est = KMeans( n_clusters=k )
est.fit( samples )
labels = est.labels_

fig,ax = plt.subplots()
colors = [ 'bo','ro','go','mo','yo','co','ko' ]
for i in range( k ):
    data = samples[ labels == i ]
    ax.plot( data[ :,0 ],data[ :,1 ],colors[ i ] )
plt.title( 'Clustered data, k = %i'%k )
plt.show()

fig,ax = plt.subplots()
for i in range( k ):
    data = samples[ labels == i ]
    ax.plot( data[ :,0 ],data[ :,1 ],colors[ i ] )
for index,mrkr in enumerate( [ 'bx','rx','gx' ] ):
    data = samples[ index*n:( index+1 )*n,: ]
    ax.plot( data[ :,0 ],data[ :,1 ],mrkr )
plt.title( 'Clustered data, k = %i'%k )
plt.show()

Try $k \in \{ 4..7 \}$ to explore how the model responds.  (The colour vector doesn't contain more entries than seven however.)

To get the cluster centroids, use `est.cluster_centers_`.

To specify the number of cluster centers, use `est = KMeans( n_cluster=k)`

To predict the association of new values, use `est.predict()`:

In [ ]:
X,Y = np.meshgrid( np.linspace( samples[ :,0 ].min(),samples[ :,0 ].max() ),np.linspace( samples[ :,1 ].min(),samples[ :,1 ].max() ) )
X = X.ravel()
X.shape = len( X ),1 # same as X = X[:,np.newaxis]
Y = Y.ravel()
Y.shape = len( Y ),1
datapts = np.concatenate((X,Y), axis=1)
C = est.predict(datapts)

fig,ax = plt.subplots()
colors = [ 'bo','ro','go','mo','yo','co','ko' ]
for i in range( k ):
    data = samples[ labels == i ]
    ax.plot( data[ :,0 ],data[ :,1 ],colors[ i ] )
colors = [ 'b+','r+','g+','m+','y+','c+','k+' ]
for i in range( len( C ) ):
    ax.plot( X[ i ],Y[ i ],colors[ C[ i ] ] )
plt.title( 'Clustered data, k = %i'%k )
plt.show()


###  Evaluation:  Silhouette method

Since any $k$ yields results, we need a way to determine which set of clusters is the "best", by whatever metric.  For instance, if outside criteria dictate some number of categories, then $k$ is fixed by the external factors.  But if we are mining data abstractly, how do we distinguish the goodness of fit for several $k$s?

The silhouette method assigns a figure of merit for the separation distance between clusters.  Silhouette coefficients are in the range $[ -1,+1 ]$; positive values indicate samples are far from cluster boundaries, values near zero indicate samples are near or on boundaries, and negative values indicate that the samples may have been assigned to the wrong cluster.  (In other words, a value closer to +1 is indicative of a better fit for the particular value of $k$.)  (Note that most of the following block is plotting code.)

In [ ]:
from sklearn.metrics import silhouette_samples, silhouette_score

k_candidates = list( range( 2,8 ) )

s_values = np.zeros( ( len( k_candidates ),1 ) )

for k in k_candidates:
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_size_inches(18, 7)

    # silhouette plot
    ax1.set_xlim( [ -1,1 ] )
    ax1.set_ylim( [ 0,len( samples )+( k+1 )*10 ] )

    est = KMeans( n_clusters=k )
    C_labels = est.fit_predict( samples )

    # average silhouette score per k
    silhouette_avg = silhouette_score( samples,C_labels )
    print( "k = %d yields average silhouette_score = %f."%( k,silhouette_avg ) )

    # silhouette scores for each sample
    sample_silhouette_values = silhouette_samples( samples,C_labels )

    ## plot of silhouette scores
    y_lower = 10
    for i in range( k ):
        # aggregate silhouette scores for samples in cluster i
        ith_cluster_silhouette_values = sample_silhouette_values[ C_labels == i ]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[ 0 ]
        y_upper = y_lower + size_cluster_i

        color = cm.get_cmap("Spectral")( float( i ) / k )
        ax1.fill_betweenx( np.arange( y_lower,y_upper ),0,ith_cluster_silhouette_values,facecolor=color,edgecolor=color,alpha=0.7 )

        # label by cluster number
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        # compute y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    ax1.set_title( "The silhouette plot for the various clusters." )
    ax1.set_xlabel( "The silhouette coefficient values" )
    ax1.set_ylabel( "Cluster label" )

    # average silhouette score as vertical line
    ax1.axvline( x=silhouette_avg,color='red',linestyle='--' )

    ax1.set_yticks( [] )    # clear y-axis labels/ticks
    ax1.set_xticks( np.linspace( -1,1,11 ) )

    ## plot of clusters 
    colors = cm.get_cmap("Spectral")( C_labels.astype( float ) / k )
    ax2.scatter( samples[ :,0 ],samples[ :,1 ],marker='.',s=30,lw=0.1, edgecolor='k',alpha=0.7,c=colors )

    # label clusters with white circles and numbers
    centers = est.cluster_centers_
    ax2.scatter( centers[ :,0 ],centers[ :,1 ],marker='o',c="white",alpha=1,s=200 )
    for i,c in enumerate( centers ):
        ax2.scatter( c[ 0 ],c[ 1 ],marker='$%d$'%i,alpha=1,s=50 )

    ax2.set_title( "The visualization of the clustered data." )
    ax2.set_xlabel( "Feature space for the 1st feature" )
    ax2.set_ylabel( "Feature space for the 2nd feature" )

    plt.suptitle( ( "Silhouette analysis for KMeans clustering on sample data with n_clusters = %d"%k ),fontsize=14,fontweight='bold' )
    plt.show()

### Evaluation:  Gap statistics method

The gap statistics method similarly assigns a figure of merit for the separation distance between clusters.  The gap statistic method seeks to maximize the gap statistic, or the spacing (gap) between clusters.  Since a `scikit-learn`-native gap statistic method has not been implemented, we will use [Miles Granger's implementation](https://github.com/milesgranger/gap_statistic).

The highest value is considered optimal.

In [ ]:
!pip install --upgrade gap-stat

In [ ]:
from gap_statistic import OptimalK
optimalK = OptimalK()

n_clusters = optimalK( datapts,cluster_array=np.arange( 1,15 ) )
print( optimalK.gap_df )
plt.plot( optimalK.gap_df.n_clusters,optimalK.gap_df.gap_value,linewidth=3 )
for c in range( 1,len( optimalK.gap_df )+1 ):
    plt.scatter( c,optimalK.gap_df.gap_value[ c-1 ],s=250,c='r' )
plt.xlabel('Cluster Count')
plt.ylabel('Gap Value')
plt.title('Gap Values by Cluster Count')
plt.show()

In this case, the two methods may not be in agreement that three clusters best describe the underlying data set.  The silhouette method "correctly" identifies three clusters, but the gap statistic method also favors the more concrete separations possible at higher cluster counts.  Unless there is a reason to expect a large number of clusters, though, one would reasonably bound the search space to a smaller subset of cluster counts.

In [ ]:
from gap_statistic import OptimalK
optimalK = OptimalK()

n_clusters = optimalK( datapts,cluster_array=np.arange( 1,7 ) )
print( optimalK.gap_df )
plt.plot( optimalK.gap_df.n_clusters,optimalK.gap_df.gap_value,linewidth=3 )
for c in range( 1,len( optimalK.gap_df )+1 ):
    plt.scatter( c,optimalK.gap_df.gap_value[ c-1 ],s=250,c='r' )
plt.xlabel('Cluster Count')
plt.ylabel('Gap Value')
plt.title('Gap Values by Cluster Count')
plt.show()

$k$-means clustering is a crude but easily implemented heuristic, and sometimes does surprisingly well.  A brief analysis of its practical limits is available with [the `scikit-learn` documentation](http://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_assumptions.html#sphx-glr-auto-examples-cluster-plot-kmeans-assumptions-py).

-   Explore the [clustering gallery](http://scikit-learn.org/stable/auto_examples/index.html#cluster-examples) for `scikit-learn`.

###  Color-Space Segmentation Using K-Means Clustering for Satellite Image Analysis

Color is an extremely sophisticated notion.  It unifies the physical wavelength of light, the biological expression of cell distribution and pigment receptors in the eye, the neurological interpretation of the resulting optic signal, and the psychological factors of culture and perception<sup>[[ref]](https://en.wikipedia.org/wiki/Opponent_process)</sup>.  Unsurprisingly, there are a lot of different ways to represent colors.  Color spaces represent different colors according to essentially different orthogonal bases.  For instance, you’re probably familiar with $RGB$ and $CMYK$ color spaces.

<table>
<tr>
<td><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c2/AdditiveColor.svg/240px-AdditiveColor.svg.png" /></td>
<td><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/19/SubtractiveColor.svg/240px-SubtractiveColor.svg.png" /></td>
</tr>
</table>

But there are many others, such as [$Lab$](https://en.wikipedia.org/wiki/Lab_color_space) and [$XYZ$](https://en.wikipedia.org/wiki/CIE_1931_color_space).  These color spaces do not necessarily cover the same range of [perceptible colors]() (or [imperceptible ones!](https://en.wikipedia.org/wiki/Impossible_color)), but transformations between spaces [can still be defined](http://www.brucelindbloom.com/Math.html).  We will use this below to convert between $RGB$ and $Lab$.

$Lab$ was designed to replicate human vision, and exploits the fact that in a sense there are [four fundamental colors](https://en.wikipedia.org/wiki/Opponent_process) that the human eye can perceive:  a red-green axis $a$ and a blue-yellow axis $b$.  Adding luminosity $L$ to these chromaticity axes yields a three-parameter color space that is actually more expressive than can be represented by $RGB$ triplets.

#### Converting from RGB to Lab

In [ ]:
!pip install colormath

In [ ]:
import numpy as np
import scipy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.feature_extraction import image
from sklearn.cluster import spectral_clustering
from sklearn.cluster import KMeans
%matplotlib inline

from colormath.color_objects import LabColor, sRGBColor
from colormath.color_conversions import convert_color
rgbize = np.vectorize(sRGBColor)
convertize = np.vectorize(convert_color)

# Inspired by an example at http://www.mathworks.com/help/images/examples/color-based-segmentation-using-k-means-clustering.html
# Read image and convert it from RGB to Lab representation.
from pylab import imread, imshow, gray, mean
sat_rgb = imread('./img/satellite.png')
plt.figure()
plt.xticks([])
plt.yticks([])
imshow(sat_rgb)
sat_rgb_cs = rgbize(sat_rgb[:,:,0],sat_rgb[:,:,1],sat_rgb[:,:,2])
sat_lab_cs = convertize(sat_rgb_cs, LabColor)

sat_lab = np.ones((sat_lab_cs.shape[0], sat_lab_cs.shape[1], 4))
# Recentering and Scaling
for i in range(sat_lab_cs.shape[0]):
    for j in range(sat_lab_cs.shape[1]):
        sat_lab[i,j,0] = sat_lab_cs[i,j].lab_l/200+100 #rgb_r
        sat_lab[i,j,1] = sat_lab_cs[i,j].lab_a/200+100 #rgb_g
        sat_lab[i,j,2] = sat_lab_cs[i,j].lab_b/200+100 #rgb_b

#### Visualizing different colorspaces
Now that we've converted the satellite image from RGB to Lab, we can look at the different color channels of the image in both colorspaces.

In [ ]:
mpl.rcParams['figure.figsize']=(15,3)

# Full-color image
plt.figure()
plt.subplot(1,4,1)
plt.xticks([])
plt.yticks([])
plt.xlabel(r'$RGB$ Full-Color Image')
imshow(sat_rgb)

# Red channel
cdict = {'red': ((0.0, 0.0, 0.0),
                 (1.0, 1.0, 1.0)),
         'green': ((0.0, 0.0, 0.0),
                   (1.0, 0.0, 0.0)),
         'blue': ((0.0, 0.0, 0.0),
                  (1.0, 0.0, 0.0))}
BkRd = mpl.colors.LinearSegmentedColormap('BkRd',cdict,256)
plt.subplot(1,4,2)
plt.xticks([])
plt.yticks([])
plt.xlabel(r'$R$')
imshow(sat_rgb[:,:,0], cmap = BkRd)

# Green channel
cdict = {'red': ((0.0, 0.0, 0.0),
                 (1.0, 0.0, 0.0)),
         'green': ((0.0, 0.0, 0.0),
                   (1.0, 1.0, 1.0)),
         'blue': ((0.0, 0.0, 0.0),
                  (1.0, 0.0, 0.0))}
BkGn = mpl.colors.LinearSegmentedColormap('BkGn',cdict,256)
plt.subplot(1,4,3)
plt.xticks([])
plt.yticks([])
plt.xlabel(r'$G$')
imshow(sat_rgb[:,:,1], cmap = BkGn)

# Blue Channel
cdict = {'red': ((0.0, 0.0, 0.0),
                 (1.0, 0.0, 0.0)),
         'green': ((0.0, 0.0, 0.0),
                   (1.0, 0.0, 0.0)),
         'blue': ((0.0, 0.0, 0.0),
                  (1.0, 1.0, 1.0))}
BkBl = mpl.colors.LinearSegmentedColormap('BkBl',cdict,256)
plt.subplot(1,4,4)
plt.xticks([])
plt.yticks([])
plt.xlabel(r'$B$')
imshow(sat_rgb[:,:,2], cmap = BkBl)

# Composite Lab image as RGB
plt.figure()
plt.subplot(1,4,1)
imshow(sat_lab)
plt.xticks([])
plt.yticks([])
plt.xlabel(r'Composite $Lab$ as $RGB$')

# L - luminosity
plt.subplot(1,4,2)
plt.xticks([])
plt.yticks([])
plt.xlabel(r'$L$')
imshow(sat_lab[:,:,0], cmap = cm.Greys_r)

# a - red-green axis
cdict = {'red': ((0.0, 0.0, 0.0),
                 (1.0, 1.0, 1.0)),
         'green': ((0.0, 1.0, 1.0),
                   (1.0, 0.0, 0.0)),
         'blue': ((0.0, 0.0, 0.0),
                  (1.0, 0.0, 0.0))}
RdGr = mpl.colors.LinearSegmentedColormap('RdGr',cdict,256)
plt.subplot(1,4,3)
plt.xticks([])
plt.yticks([])
plt.xlabel(r'$a$')
imshow(sat_lab[:,:,1], cmap = RdGr)

# b - blue-yellow axis
cdict = {'red': ((0.0, 0.0, 0.0),
                 (1.0, 1.0, 1.0)),
         'green': ((0.0, 0.0, 0.0),
                   (1.0, 1.0, 1.0)),
         'blue': ((0.0, 1.0, 1.0),
                  (1.0, 0.0, 0.0))}
BlYw = mpl.colors.LinearSegmentedColormap('BlYw',cdict,256)
plt.subplot(1,4,4)
plt.xticks([])
plt.yticks([])
plt.xlabel(r'$b$')
imshow(sat_lab[:,:,2], cmap = BlYw)

### Your Turn:  Clustering to Segment Images in the Lab Colorspace

Now that we've converted to the Lab colorspace, we can use clustering to segment satellite images into different colors.  In the Lab colorspace, the color information is contained in the a and b channels.  Pixels that have a and b channel coordinates that are far apart represent colors that are more different from one another.  So, to segment images we'll apply K-means clustering on points in 2D space (where the 2D space are the a and b channel coordinates).

### Your Tasks:
#### Task #1
Use one of the two techniques described above (silhouette method or gap statistic) to identify to identify the two most likely numbers of clusters between 2 and 9.
   <br>**Hints**:
   * If you get a memory error when trying to run silhouette, you may need to run it on on smaller subset of the pixels.  One option would be to downsample by a factor of 3 in each dimension (so running on 1/9 of the number of pixels.  Recall with numpy slicing you can specify start, stop, and step for each dimension (example for 1d array:  `arr[start:stop:step]`).  What value of step would you use to take every third pixel?
   * Before running $k$-means (part of silhouette) or calling the gap statistic, the data points will need to be in the right shape.  See the hint for task #2.
   
#### Task #2
Use sklearn.cluster.KMeans to perform clustering on the color coordinates in the a and b channels of the Lab colorspace for the 2 values of $k$ identified in task #1. When calling $k$-means, pass `n_init=10` to run 10x to avoid local minima.
  <br>**Hint:**
  * In order to use `sklearn.cluster.KMeans`, you need a 2D numpy array where each row is a point in the 2D colorspace (a and b channels).  You'll need to select the correct channels from the `sat_lab` array and then reshape so the array of "points" are in the correct shape.
    
#### Task #3
Use the function `show_segmentations()` given below to illustrate the image segmentations resulting from the clustering performed in task #2 for the 2 values of $k$ identified in task #1.
  <br>**Hint:**
  * `show_segmentations()` takes an argument called `pixel_labels`, which is a 2D array in the shape of the original image, where each entry is the label for each pixel.  Per the hint for task #2, to call $k$-means, the datapoints (a and b coordinates for each pixel) must be a 2D array, with 1 row per pixel.  So, you'll need to be sure to reshape the pixel labels resulting from $k$-means before calling `show_segmentations()`.
    

In [ ]:
def show_segmentations(image_rgb, n_colors, pixel_labels):
    '''
    Illustrate segmentations of original image by color cluster.
    
    Args:
        image_rgb (numpy array): RGB image array
        n_colors (int): number of colors
        pixel_labels (numpy array): 3d array indicating pixel labels
    '''
    # Show pixel labeling
    mpl.rcParams['figure.figsize']=(5,5)
    plt.figure
    plt.xticks([])
    plt.yticks([])
    imshow(pixel_labels, cmap = cm.Blues_r)
    plt.tile('Image Segmentation')
    plt.show()
    
    # Show each individual segment
    mpl.rcParams['figure.figsize']=(16,8)
    img_seg = np.zeros((n_colors,image_rgb.shape[0],image_rgb.shape[1],image_rgb.shape[2]))

    for k in range(n_colors):
        color_index = np.where(pixel_labels == k)
        img_seg[k,color_index[0],color_index[1]] = img_rgb[np.where(pixel_labels == k)]

        plt.subplot(2,np.ceil(n_colors/2),k+1)
        plt.xticks([])
        plt.yticks([])
        imshow(img_seg[k])
    plt.tight_layout()
    plt.suptitle('Individual Image Segments')
    plt.show()

#### Applying silhouette method to determine $k$

In [ ]:
k_candidates = list( range( 2,10 ) )

s_values = np.zeros( ( len( k_candidates ),1 ) )

ab = sat_lab[:,:,1:3]
# Downsample by 2 in each dimension to avoid memory error
ab = sat_lab[:,:,1:3][::2,::2]
ab = np.reshape(ab, (ab.shape[0]*ab.shape[1],2))

for k in k_candidates:
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_size_inches(18, 7)

    # silhouette plot
    ax1.set_xlim( [ -1,1 ] )
    ax1.set_ylim( [ 0,len( ab )+( k+1 )*10 ] )

    est = KMeans( n_clusters=k )
    C_labels = est.fit_predict( ab )

    # average silhouette score per k
    silhouette_avg = silhouette_score( ab,C_labels )
    print( "k = %d yields average silhouette_score = %f."%( k,silhouette_avg ) )

    # silhouette scores for each sample
    sample_silhouette_values = silhouette_samples( ab,C_labels )

    ## plot of silhouette scores
    y_lower = 10
    for i in range( k ):
        # aggregate silhouette scores for samples in cluster i
        ith_cluster_silhouette_values = sample_silhouette_values[ C_labels == i ]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[ 0 ]
        y_upper = y_lower + size_cluster_i

        color = cm.get_cmap("Spectral")( float( i ) / k )
        ax1.fill_betweenx( np.arange( y_lower,y_upper ),0,ith_cluster_silhouette_values,facecolor=color,edgecolor=color,alpha=0.7 )

        # label by cluster number
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        # compute y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    ax1.set_title( "The silhouette plot for the various clusters." )
    ax1.set_xlabel( "The silhouette coefficient values" )
    ax1.set_ylabel( "Cluster label" )

    # average silhouette score as vertical line
    ax1.axvline( x=silhouette_avg,color='red',linestyle='--' )

    ax1.set_yticks( [] )    # clear y-axis labels/ticks
    ax1.set_xticks( np.linspace( -1,1,11 ) )

    ## plot of clusters 
    colors = cm.get_cmap("Spectral")( C_labels.astype( float ) / k )
    ax2.scatter( ab[ :,0 ],ab[ :,1 ],marker='.',s=30,lw=0.1, edgecolor='k',alpha=0.7,c=colors )

    # label clusters with white circles and numbers
    centers = est.cluster_centers_
    ax2.scatter( centers[ :,0 ],centers[ :,1 ],marker='o',c="white",alpha=1,s=200 )
    for i,c in enumerate( centers ):
        ax2.scatter( c[ 0 ],c[ 1 ],marker='$%d$'%i,alpha=1,s=50 )

    ax2.set_title( "The visualization of the clustered data." )
    ax2.set_xlabel( "Feature space for the 1st feature" )
    ax2.set_ylabel( "Feature space for the 2nd feature" )

    plt.suptitle( ( "Silhouette analysis for KMeans clustering on sample data with n_clusters = %d"%k ),fontsize=14,fontweight='bold' )
    plt.show()

#### Applying gap statistic

In [ ]:
# Running Gap Statistic  on 
optimalK = OptimalK()

ab = sat_lab[:,:,1:3]
ab = np.reshape(ab,(sat_lab.shape[0]*sat_lab.shape[1],2))

n_clusters = optimalK( ab, cluster_array=np.arange( 1,10 ) )
print( optimalK.gap_df )
plt.plot( optimalK.gap_df.n_clusters,optimalK.gap_df.gap_value,linewidth=3 )
for c in range( 1,len( optimalK.gap_df )+1 ):
    plt.scatter( c,optimalK.gap_df.gap_value[ c-1 ],s=250,c='r' )
plt.xlabel('Cluster Count')
plt.ylabel('Gap Value')
plt.title('Gap Values by Cluster Count')
plt.show()

#### Visualizing Results
The 2 most likely values of $k$ based on the parameter choice methods were 2 and 6.  The below section blocks run $k$-means for each of these values of $k$ and calls the visualization function to illustrate the results.

In [ ]:
# Classify each color into clusters using the K-means algorithm.
from sklearn.cluster import KMeans
ab = sat_lab[:,:,1:3];
ab = np.reshape(ab,(sat_lab.shape[0]*sat_lab.shape[1],2));

# How many major colors do you perceive?
n_colors = 2

# Cluster, repeating 10x to avoid local minima.
kmeans = KMeans(n_clusters=n_colors, n_init=10)
cluster_index = kmeans.fit_predict(ab)

# Classify pixels by K-means cluster.
pixel_labels = np.reshape(cluster_index,(sat_lab.shape[0],sat_lab.shape[1]))
show_segmentations(sat_lab, n_colors, pixel_labels)

In [ ]:
# Classify each color into clusters using the K-means algorithm.
from sklearn.cluster import KMeans
ab = sat_lab[:,:,1:3];
ab = np.reshape(ab,(sat_lab.shape[0]*sat_lab.shape[1],2));

# How many major colors do you perceive?
n_colors = 6

# Cluster, repeating 10x to avoid local minima.
kmeans = KMeans(n_clusters=n_colors, n_init=10)
cluster_index = kmeans.fit_predict(ab)

# Classify pixels by K-means cluster.
pixel_labels = np.reshape(cluster_index,(sat_lab.shape[0],sat_lab.shape[1]))
show_segmentations(sat_lab, n_colors, pixel_labels)

The above lesson was developed by Erhu Du, Jane Lee, and Neal Davis for Computational Science and Engineering at the University of Illinois.  Development was supported by a grant from MathWorks, Inc.